In [ ]:
import numpy as np
import pandas as pd

from dotenv import load_dotenv

from panel import llm, paths
from panel.llm import run_llm_batch
from panel.analysis import aggregate, plots
from panel.run import RunSpec, ResultStore
from panel.topic import load_topic, make_response_models
from datasets.sod.describe import region_in, region_of

load_dotenv()

## Experiment Setup

In [ ]:
def citizen_profile(row):
    return "Jsem obyvatel/ka " + region_of(row["region"]) + "."

def direct_forecast(row):
    return "Účast a výsledek voleb do poslanecké sněmovny " + region_in(row["region"]) + " v roce 2021 budou:"

In [ ]:
# Run setup - one row per prompt, REPEATS rows per region.

REPEATS = 10
REGIONS_ALL = pd.read_csv(paths.DATASETS_DIR / "regions" / "regions.csv")["region"].tolist()
regions = pd.DataFrame({"region": [r for r in REGIONS_ALL for _ in range(REPEATS)]})

# 150 synthetic rows built right here, so they describe themselves right here - no template
# file and no prompt table, because there is no cleanup pass to render one in. The spec
# still just names a column, same as the survey runs.
regions["latent_citizen_v1"] = regions.apply(citizen_profile, axis=1)
regions["latent_direct_v1"] = regions.apply(direct_forecast, axis=1)

TOPIC = load_topic("election_2021", paths.TOPICS_DIR)
VProb, VPick = make_response_models(TOPIC)

BASE = RunSpec(
    name="latent_citizen",
    topic=TOPIC.name,
    data=regions,
    description="latent_citizen_v1",
    question=TOPIC.question,
    instructions=TOPIC.instructions,
    client_type=llm.OpenAIClient,
    model="gpt-4.1-nano",
    response_model=VProb,
    temperature=0,
)

SPECS = {
    "citizen": BASE,
    "direct":  BASE.variant(name="latent_direct", description="latent_direct_v1", question=""),
}

SPEC = SPECS["citizen"]

STORE = ResultStore(paths.RESULTS_DIR, SPEC.topic)
CLIENT = SPEC.make_client()

In [ ]:
SPEC.prompt(SPEC.data.iloc[0])

## Election Preferences Experiment

In [ ]:
NORMALIZE = TOPIC.normalize_groups

voting_results, skipped = run_llm_batch(
    data=SPEC.rows(),
    prompt_creator=SPEC.prompt,
    response_model=SPEC.response_model,
    model=SPEC.model, temperature=SPEC.temperature,
    client=CLIENT, instructions=SPEC.instructions
)
print(f"Processed: {len(voting_results)}, skipped: {len(skipped)}")

STORE.save(SPEC, voting_results, normalize=NORMALIZE)
paths = STORE.save(SPEC, voting_results, normalize=NORMALIZE, timestamped=False)
paths

In [ ]:
results_df = STORE.frame(SPEC.stem)

latent_results = []
for region in REGIONS_ALL:
    region_rows = SPEC.data.index[SPEC.data["region"] == region]
    region_result = aggregate.aggregate_results_from_df(results_df.loc[region_rows], TOPIC)
    region_result["region"] = region
    latent_results.append(region_result)

latent_results_df = pd.DataFrame(latent_results)
latent_results_df.to_csv(STORE.analysis(f"by_region_{SPEC.stem}.csv"))
latent_results_df

## Visualization

In [ ]:
latent_results_df = pd.read_csv(STORE.analysis(f"by_region_{SPEC.stem}.csv"), index_col=0)
actual_results_df = TOPIC.load_reference()

for region in REGIONS_ALL:
    pred = latent_results_df[latent_results_df["region"] == region].iloc[0]
    actual = actual_results_df[actual_results_df["region"] == region].iloc[0]
    fig = plots.visualize_comprehensive_results(
        pred,
        actual,
        list(TOPIC.options), TOPIC.turnout,
        model_name=f"{SPEC.model} {region}", actual_is_claimed=False
    )

In [ ]:
citizen = pd.read_csv(STORE.analysis(f"by_region_{SPECS['citizen'].stem}.csv"), index_col=0)
direct = pd.read_csv(STORE.analysis(f"by_region_{SPECS['direct'].stem}.csv"), index_col=0)

for region in REGIONS_ALL:
    pred = citizen[citizen["region"] == region].iloc[0]
    other = direct[direct["region"] == region].iloc[0]
    fig = plots.visualize_comprehensive_results(
        pred,
        other,
        list(TOPIC.options), TOPIC.turnout,
        model_name=f"{SPECS['citizen'].model} {region}", actual_is_claimed=False
    )